# Exemple d'Utilisation - Pipeline MLOps

## 1. Importer les modules

In [29]:
from utils.pipeline import MLPipeline
from utils.data_loader import load_config
import logging

logging.basicConfig(level=logging.DEBUG, format='%(asctime)s - %(levelname)s - %(message)s')

In [30]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(".env"), override=True)      # config
load_dotenv(find_dotenv(".env.secrets"), override=True)   # secrets only useful on local environment

True

## 2. Initialiser le pipeline

In [31]:
print("\n>>> Initialisation du pipeline...\n")
pipeline = MLPipeline(config_path="utils/config.yaml")
#file_path = "data/raw/extract_csv_engie_dataset.csv"
file_path = "data/raw/extract_csv_enedis_dataset_500000.csv"

INFO:utils.pipeline:Pipeline MLOps initialisé



>>> Initialisation du pipeline...



## 3. Exécuter le pipeline complet

In [32]:
# Note: Remplacer par votre fichier de données
#print("\n>>> Lancement du pipeline complet...\n")
#success = pipeline.run_full_pipeline(file_path)

#if success:
#    print("\n✓ Pipeline exécuté avec succès!")
#    print(f"\nMétriques finales: {pipeline.metrics}")
#else:
#    print("\n✗ Erreur lors de l'exécution du pipeline")

## 4. Alternative: Exécuter étape par étape

In [33]:
print("\n" + "="*50)
print("EXÉCUTION ÉTAPE PAR ÉTAPE (alternative)")
print("="*50)

pipeline2 = pipeline  # Cloner le pipeline pour une exécution étape par étape


EXÉCUTION ÉTAPE PAR ÉTAPE (alternative)


In [34]:
print("\n1. Chargement des données...")
pipeline2.step_1_load_data(file_path)

INFO:utils.pipeline:=== ÉTAPE 1: CHARGEMENT DES DONNÉES ===



1. Chargement des données...
Données chargées avec succès: data/raw/extract_csv_enedis_dataset_500000.csv
Forme des données: (550000, 11)
Colonnes: ['secteur_activite', 'plage_de_puissance_souscrite', 'region', 'nb_points_soutirage', 'date', 'ville', 'nom_vacances', 'temperature_2m_mean', 'relative_humidity_mean', 'precipitation_sum', 'total_energie_soutiree_wh\r']
Types: {'secteur_activite': dtype('O'), 'plage_de_puissance_souscrite': dtype('O'), 'region': dtype('O'), 'nb_points_soutirage': dtype('int64'), 'date': dtype('O'), 'ville': dtype('O'), 'nom_vacances': dtype('O'), 'temperature_2m_mean': dtype('float64'), 'relative_humidity_mean': dtype('float64'), 'precipitation_sum': dtype('float64'), 'total_energie_soutiree_wh\r': dtype('O')}
Head:   secteur_activite plage_de_puissance_souscrite                region  \
0  S1: Agriculture             P1: ]36-120] kVA  Auvergne-Rhône-Alpes   
1  S1: Agriculture             P1: ]36-120] kVA  Auvergne-Rhône-Alpes   
2  S1: Agriculture       

True

In [35]:
print("\n2. Validation des données...")
pipeline2.step_2_validate_data(save_report=True)

INFO:utils.pipeline:=== ÉTAPE 2: VALIDATION DES DONNÉES ===



2. Validation des données...


INFO:utils.data_validator:Validation: Valeurs manquantes=13.34%, Doublons=34.83%
INFO:utils.pipeline:Résultats de validation: False
INFO:utils.data_validator:Rapport simple créé: outputs/evidently_report.html


np.False_

In [36]:
print("\n3. Préparation et Prétraitement des données...")
print("   - Split train/test")
print("   - Détection automatique des types de colonnes")
print("   - Imputation des valeurs manquantes")
print("   - Normalisation des données numériques")
print("   - Encodage des variables catégories")

pipeline2.step_3_prepare_data()

if pipeline2.preprocessor and pipeline2.feature_names:
    print(f"\n✓ Données préparées et prétraitées!")
    print(f"  Nombre de features après preprocessing: {len(pipeline2.feature_names)}")
    print(f"  Forme train: {pipeline2.X_train.shape}")
    print(f"  Forme test: {pipeline2.X_test.shape}")

INFO:utils.pipeline:=== ÉTAPE 3: PRÉPARATION ET PRÉTRAITEMENT ===
INFO:utils.pipeline:  → Division train/test...



3. Préparation et Prétraitement des données...
   - Split train/test
   - Détection automatique des types de colonnes
   - Imputation des valeurs manquantes
   - Normalisation des données numériques
   - Encodage des variables catégories


INFO:utils.pipeline:  ✓ Split effectué: train=(65150, 10), test=(16288, 10)
INFO:utils.pipeline:  → Prétraitement avancé (imputation, scaling, encoding)...
INFO:utils.data_preparation:=== PRÉPARATION DES DONNÉES ===
INFO:utils.data_preparation:
1️⃣ Détection des types de features...
INFO:utils.data_preparation:🔍 Features numériques détectées: ['nb_points_soutirage', 'temperature_2m_mean', 'relative_humidity_mean', 'precipitation_sum']
INFO:utils.data_preparation:🔍 Features catégories détectées: ['secteur_activite', 'plage_de_puissance_souscrite', 'region', 'date', 'ville', 'nom_vacances']
INFO:utils.data_preparation:
2️⃣ Création du preprocessor...
INFO:utils.data_preparation:✓ Pipeline numériques créé: Imputer(mean) + Scaler
INFO:utils.data_preparation:✓ Pipeline catégories créé: OneHotEncoder
INFO:utils.data_preparation:✓ ColumnTransformer créé
INFO:utils.data_preparation:
3️⃣ Transformation du training set...


Données après nettoyage: (91632, 11)
Features shape: (91632, 10), Target shape: (91632,)
Attention: Conversion numérique partiellement échouée: could not convert string to float: 'S1: Agriculture'
Ensemble d'entraînement: (65150, 10)
Ensemble de test: (16288, 10)


INFO:utils.data_preparation:  ✓ Shape: (65150, 36)
INFO:utils.data_preparation:
4️⃣ Transformation du test set...
INFO:utils.data_preparation:  ✓ Shape: (16288, 36)
INFO:utils.data_preparation:
5️⃣ Récupération des noms de features...
INFO:utils.data_preparation:  ✓ 4 features au total
INFO:utils.data_preparation:
✓ Préparation des données terminée

INFO:utils.pipeline:  ✓ Prétraitement effectué: (65150, 36) (train), (16288, 36) (test)
INFO:utils.pipeline:  ✓ Features après preprocessing: 4



✓ Données préparées et prétraitées!
  Nombre de features après preprocessing: 4
  Forme train: (65150, 36)
  Forme test: (16288, 36)


In [37]:
print("\n4. Entraînement du modèle...")
print("   (sur les données préparées/préprocessées)")
print(set(pipeline2.y_train))
pipeline2.step_4_train_model()


if pipeline2.model:
    print(f"✓ Modèle entraîné: {type(pipeline2.model).__name__}")

INFO:utils.pipeline:=== ÉTAPE 4: ENTRAÎNEMENT DU MODÈLE ===
INFO:utils.model:X_train shape: (65150, 36), y_train shape: (65150,)
INFO:utils.model:Valeurs uniques dans y: 41754
INFO:utils.model:Modèle de régression Random Forest créé



4. Entraînement du modèle...
   (sur les données préparées/préprocessées)
{0.0, 6422529.0, 4849666.0, 131083.0, 3932172.0, 1048588.0, 2097166.0, 82051084.0, 228458512.0, 1441811.0, 52953107.0, 6029333.0, 13762583.0, 189530136.0, 2228250.0, 5242908.0, 63438879.0, 58327071.0, 393250.0, 524322.0, 92536875.0, 39452716.0, 55967790.0, 17694767.0, 87294000.0, 17432625.0, 1572916.0, 18481209.0, 4718653.0, 5505086.0, 7864383.0, 48103511.0, 262232.0, 393309.0, 64880737.0, 131169.0, 262250.0, 1179758.0, 1179765.0, 1441916.0, 6160513.0, 34996356.0, 17825926.0, 16121990.0, 655500.0, 1179789.0, 62128274.0, 1310872.0, 49414298.0, 56885402.0, 9699484.0, 655514.0, 4718750.0, 655524.0, 375390374.0, 4194471.0, 1048750.0, 32506031.0, 266862769.0, 131250.0, 19267762.0, 9044147.0, 1179833.0, 23593146.0, 262333.0, 14942400.0, 9961665.0, 131264.0, 17694916.0, 15335621.0, 393416.0, 354418889.0, 39846090.0, 41812173.0, 9175250.0, 1966292.0, 26083546.0, 17432804.0, 655590.0, 1573101.0, 2490608.0, 22675699.0, 39

INFO:utils.model:Modèle entraîné avec succès


✓ Modèle entraîné: RandomForestRegressor


In [38]:
print("\n5. Évaluation du modèle...")
pipeline2.step_5_evaluate_model()

if pipeline2.metrics:
    print("\n--- RÉSULTATS ---")
    for key, value in pipeline2.metrics.items():
        print(f"  {key}: {value:.4f}")

INFO:utils.pipeline:=== ÉTAPE 5: ÉVALUATION DU MODÈLE ===



5. Évaluation du modèle...


INFO:utils.model:Régression - RMSE: 9816973.979, R²: 0.975



--- RÉSULTATS ---
  mse: 96372978113601.7969
  rmse: 9816973.9795
  r2: 0.9752


In [39]:
print("\n6. Monitoring de la performance...")
pipeline2.step_6_monitor_performance()

INFO:utils.pipeline:=== ÉTAPE 6: MONITORING DE LA PERFORMANCE ===



6. Monitoring de la performance...


INFO:utils.performance_monitor:Pas de drift détecté
INFO:utils.pipeline:Résultats du monitoring: Drift=False


True

In [40]:
print("\n7. Logging avec MLflow...")
pipeline2.step_7_log_with_mlflow()

INFO:utils.pipeline:=== ÉTAPE 7: LOGGING AVEC MLFLOW ===
INFO:utils.mlflow_tracker:Tracking URI configuré: https://jenedai-mlflow.hf.space/



7. Logging avec MLflow...


INFO:utils.mlflow_tracker:Run MLflow démarrée - Expérience: enedis_project
INFO:utils.mlflow_tracker:Paramètres enregistrés: {'test_size': 0.2, 'random_state': 42, 'model_type': 'random_forest'}
INFO:utils.mlflow_tracker:Métriques enregistrées: ['mse', 'rmse', 'r2']
2026/04/01 23:58:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/01 23:58:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
INFO:utils.mlflow_tracker:Modèle enregistré: model


🏃 View run overjoyed-grouse-641 at: https://jenedai-mlflow.hf.space/#/experiments/3/runs/172054c7ba15446aadf4c25b24cc1958
🧪 View experiment at: https://jenedai-mlflow.hf.space/#/experiments/3


INFO:utils.mlflow_tracker:Run MLflow terminée
INFO:utils.mlflow_tracker:Session d'entraînement enregistrée dans MLflow


True

In [41]:
print("\n8. Gestion des Stages du Modèle MLflow...")
# Métriques pour prédiction énergétique (5 jours)
# MAE = Mean Absolute Error (valeur absolue moyenne des erreurs)
# RMSE = Root Mean Squared Error (racine carrée de la moyenne des carrés)
# Accuracy = Pour certains modèles de classification
metric_keys = ["mae", "rmse", "accuracy"]
pipeline2.step_8_manage_model_stages(metric_keys=metric_keys, min_improvement=0.0)

INFO:utils.pipeline:=== ÉTAPE 8: GESTION DES ALIASES DU MODÈLE ===
INFO:utils.mlflow_tracker:Tracking URI configuré: https://jenedai-mlflow.hf.space/



8. Gestion des Stages du Modèle MLflow...


INFO:utils.pipeline:
[1/4] Récupération du run_id et enregistrement du modèle en Staging...
INFO:utils.pipeline:  Métriques à comparer (priorité): ['mae', 'rmse', 'accuracy']
INFO:utils.pipeline:  ℹ Run trouvée: 172054c7ba15446aadf4c25b24cc1958
Registered model 'enedis_model' already exists. Creating a new version of this model...
2026/04/01 23:58:55 WARNING mlflow.tracking._model_registry.fluent: Run with id 172054c7ba15446aadf4c25b24cc1958 has no artifacts at artifact path 'model', registering model based on models:/m-dc7110723833498cad61949162e2779d instead
2026/04/01 23:58:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: enedis_model, version 6
Created version '6' of model 'enedis_model'.
INFO:utils.pipeline:  ✓ Modèle enedis_model v6 enregistré
INFO:utils.pipeline:    Stage initial: None
INFO:utils.pipeline:
[2/4] État des versions avec alias 'prod'...
INFO:utils.pipeline:  ℹ Version actuelle avec alias

True

In [42]:
print("\n✓ Pipeline étape par étape terminé!")


✓ Pipeline étape par étape terminé!
